# Wprowadzenie do sieci tensorowych

Tutaj będzie zaprezentowane wprowadzenie do sieci tensorowych (TN - *ang. Tensor Networks*). Jego głównym celem jest zaprezantowanie podstawowej terminologii i narzędzi, potrzebych do zrozumienia algorytmów wykożystujących TN. 

## Tensory i zwęrzanie indeksów

Na nasze potrzeby, jako tensor będziemy rozumieli indeksowany zbiór liczb, albo inaczej mówiąc, wielowymiarową tablicę liczb. Jako **rząd** tensora będziemy rozumieli liczbę różnych indeksów (albo kolokwialnie ile "wymiarów" posiada). Tak więc wektor to jest tensor 1-rzędu,  Macierz to tensor 2-rzędu, a trójwymarowa tablica liczb to tensor 3-rzędu itd.

![TN](pictures/tensor-illustration.jpg)

*Zwężaniem indeksu* nazywamy sumę po wszystkich możliwych wartościach powtarzających się indeksów w zbiorze tensorów. Przykładowo, nnorzenie macierzy można przedstawić jako:

$$
C_{ik} = \sum_{j=1}^D A_{ij} B_{jk}.
$$

Jest to zwęrzenie indeksu $j$, czyli suma po jego wszystkich $D$ wartościach, $j = 1, \ldots, D$. Bardziej skomplikowaqny przykład:

$$
D_{ijk} = \sum_{l=1}^{D_1}\sum_{m=1}^{D_2}\sum_{n=1}^{D_3} A_{ljm} B_{iln} C_{nmk}.
$$

Warto zauważyć, że każdy indeks moze przyjmować różne wartości. 

Jak widać w powyższych przykładach, zwęrzanie indeksów daje w wyniku nowy tensor, tak samo jak przykładowo mnożenie macieży daje nową macierz. Indeksy które nie zostały zwęrzone nazywamy *otwartymi indeksami*.

Poniżej jest przedstawiony kod zwiężania tensorów.

In [4]:
# Podstawowe zwęrzanie tensorów

using LinearAlgebra

A = rand(2,3,4)
B = rand(5,2,6)
C = rand(6,4,7)


l, j, m = size(A)
i, _, n = size(B)
_, _, k = size(C)
D = zeros(i, j, k)

for ii in 1:i, jj in 1:j, kk in 1:k
    for ll in 1:l, mm in 1:m, nn in 1:n
        D[ii, jj, kk] += A[ll, jj, mm] * B[ii, ll, nn] * C[nn, mm, kk]
    end
end

println("Rozmiar tensora D: ", size(D))

Rozmiar tensora D: (5, 3, 7)


Jak widać, pisanie tych sumowań "z palca" jest bardzo nieporęczne. Na szczęście istnieją biblioteki które ułatwiają to zadanie. Jedną z nich jest `TensorOperations.jl` z bardzo przydanym makro `@tensor` 

In [ ]:
using TensorOperations

@tensor D_prim[i, j, k] := A[l, j, m] * B[i, l, n] * C[n, m, k]

println("Rozmiar tensora D': ", size(D_prim))

println("Tensor D równy tensorowi D': ", D≈D_prim)  

Rozmiar tensora D': (5, 3, 7)
Tensor D równy tensorowi D' true


## Sieci tensorowe i ich diagramy

*Sieć tensorowa* jest zbiorem tensorów w którym indeksy są zwęrzane zgodnie z pewnym schematem. Oba wcześniejsze przykłady są przykładami sieci tensorowych. Bardzo wygodnym sposobem przedstawiania TN są tzw. *diagramy sieci tensorowych*. W tych diagramach tensory są przedstawiane jako pewne kształty (najczęściej koła), a indeksy są przedstawione jako linie wychodzące z tych kształtów.

![image](pictures/tn_viz_placeholder.png)

Sieć tensorowa jest więc reprezentowana przez zbiór kształtów połączonych liniami. Linia łącząca dwa tensory reprezentuje zwęrzanie konkretnych indeksów, a linie niepołączone z niczym reprezentują otwarte indeksy.

![image](pictures/tn_con_placeholder.png)

## MPS

MPS (*ang. - Matrix Product State*) jest rodziną sieci tensorowych, gdzie tensory są ułożone w łańcuch. 

OBRAZEK

### Przykład - funkcja podziału modelu Isinga jako MPS

Dla przypomienia, funckja podziału dla modelu Isinga:

$$
Z = \sum_{s\in\mathcal{S}} e^{-\beta H(s)}
$$

gdzie $s = [s_1, \ldots s_N]$ jest stanem (konfiguracją), a $\mathcal{S}$ jest zbiorem wszystik możliwych stanów dla $N$ spinów.

In [ ]:
# WIP - model Isinga jako sieć tensorowa

using TensorOperations
using LinearAlgebra

# Parameters
β = 1.0
J = [0.5, -1.0, 0.8]         # Couplings: J₁₂, J₂₃, J₃₄
h = [0.3, -0.2, 0.1, 0.5]    # Local fields: h₁, h₂, h₃, h₄

# Spin values corresponding to indices 1 and 2
spinvals = [-1, 1]

# Build field tensors: v[i][s_index]
v = [zeros(2) for _ in 1:4]
for i in 1:4, (s_idx, s) in enumerate(spinvals)
    v[i][s_idx] = exp(β * h[i] * s)
end

# Build coupling tensors: W[i][s_i, s_{i+1}]
W = [zeros(2,2) for _ in 1:3]
for i in 1:3, (s1_idx, s1) in enumerate(spinvals), (s2_idx, s2) in enumerate(spinvals)
    W[i][s1_idx, s2_idx] = exp(β * J[i] * s1 * s2)
end

δ = Matrix{Float64}(I, 2, 2)

# Tensor contraction using Einstein notation
@tensor Z := v[1][s1] * W[1][s1, s2a] * v[2][s2a] *
             W[2][s2b, s3a] * v[3][s3a] *
             W[3][s3b, s4] * v[4][s4] *
             δ[s2a, s2b] * δ[s3a, s3b]

println("Partition function Z = $Z")

LoadError: LoadError: ArgumentError: index s2a appears more than twice in tensor contraction: var"##v[1]#237"[s1] * var"##W[1]#238"[s1, s2a] * var"##v[2]#239"[s2a] * var"##W[2]#240"[s2b, s3a] * var"##v[3]#241"[s3a] * var"##W[3]#242"[s3b, s4] * var"##v[4]#243"[s4] * δ[s2a, s2b] * δ[s3a, s3b]
in expression starting at /home/tsmierzchalski/AkademiaSztukiKwantowej/tensor_networks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X25sdnNjb2RlLXJlbW90ZQ==.jl:27

## PEPS

* Potrzeba heurystycznewgo zwęrzania
* wykorzystanie GPU

# Bibliografia
* Orús, R. (2014). A practical introduction to tensor networks: Matrix product states and projected entangled pair states. *Annals of Physics*, *349*, 117-158.
